In [10]:
!nvidia-smi
!pip install -q numba cupy-cuda12x pytest pynvml

from numba import cuda
import cupy as cp

print("CUDA доступна:", cuda.is_available())
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])


Fri Oct 16 09:09:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             29W /   70W |     153MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:

import numpy as np
from numba import cuda
@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)

    if i < a.size:
        out[i] = a[i] + b[i]
N = 1024
a = np.arange(N, dtype=np.float32)
b = np.arange(N, dtype=np.float32) * 2
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_out = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (N + 255) // 256
vector_add_1d[blocks_per_grid, threads_per_block](
    d_a, d_b, d_out
)

result = d_out.copy_to_host()

expected = a + b
np.testing.assert_allclose(
    result, expected, atol=1e-5
)

print("Task 1 PASSED!")
print("First 10 results:", result[:10])
print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)


Task 1 PASSED!
First 10 results: [ 0.  3.  6.  9. 12. 15. 18. 21. 24. 27.]
Threads per block: 256
Blocks per grid: 4


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [12]:

import numpy as np
from numba import cuda

@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        out[i] = x[i] * alpha
        i += stride

N = 1000000
alpha = 2.5

x = np.arange(N, dtype=np.float32)

d_x = cuda.to_device(x)
d_out = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = 64

vector_scale_grid_stride[
    blocks_per_grid, threads_per_block
](d_x, alpha, d_out)

result = d_out.copy_to_host()
expected = x * alpha

np.testing.assert_allclose(result, expected, atol=1e-5)

print("Task 2 PASSED!")
print("First 10 results:", result[:10])
print("Total elements:", N)
print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)
print("Total GPU threads:", threads_per_block * blocks_per_grid)


Task 2 PASSED!
First 10 results: [ 0.   2.5  5.   7.5 10.  12.5 15.  17.5 20.  22.5]
Total elements: 1000000
Threads per block: 256
Blocks per grid: 64
Total GPU threads: 16384


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [13]:

import numpy as np
from numba import cuda

@cuda.jit
def matrix_add_2d(A, B, C):
    col, row = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

M = 513
N = 1021

A = np.random.rand(M, N).astype(np.float32)
B = np.random.rand(M, N).astype(np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((M, N), dtype=np.float32)

threads_per_block = (16, 16)
blocks_per_grid = ((N + 15) // 16, (M + 15) // 16)

matrix_add_2d[blocks_per_grid, threads_per_block](
    d_A, d_B, d_C
)

result = d_C.copy_to_host()
expected = A + B

np.testing.assert_allclose(result, expected, atol=1e-5)

print("Task 3 PASSED!")
print("Matrix shape:", result.shape)
print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)
print("First 5 elements:", result[0, :5])


Task 3 PASSED!
Matrix shape: (513, 1021)
Threads per block: (16, 16)
Blocks per grid: (64, 33)
First 5 elements: [1.444449   0.9356607  0.51755774 0.87672853 1.653924  ]


In [14]:

import numpy as np
from numba import cuda

@cuda.jit
def matrix_transpose_naive(A, B):
    col, row = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]

M = 513
N = 1021

A = np.random.rand(M, N).astype(np.float32)

d_A = cuda.to_device(A)
d_B = cuda.device_array((N, M), dtype=np.float32)

threads_per_block = (16, 16)
blocks_per_grid = ((N + 15) // 16, (M + 15) // 16)

matrix_transpose_naive[blocks_per_grid, threads_per_block](
    d_A, d_B
)

result = d_B.copy_to_host()
expected = A.T

np.testing.assert_allclose(result, expected, atol=1e-5)

print("Task 4 PASSED!")
print("Original matrix shape:", A.shape)
print("Transposed matrix shape:", result.shape)
print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)
print("Original first row:", A[0, :5])
print("Transposed first column:", result[:5, 0])


Task 4 PASSED!
Original matrix shape: (513, 1021)
Transposed matrix shape: (1021, 513)
Threads per block: (16, 16)
Blocks per grid: (64, 33)
Original first row: [0.5320582  0.12975049 0.64675814 0.09903314 0.13839579]
Transposed first column: [0.5320582  0.12975049 0.64675814 0.09903314 0.13839579]


In [15]:

import numpy as np
from numba import cuda

@cuda.jit
def volume_init_3d(vol, D, H, W):
    x, y, z = cuda.grid(3)

    if x < W and y < H and z < D:
        vol[z, y, x] = z * 10000 + y * 100 + x

D = 33
H = 47
W = 65

d_vol = cuda.device_array((D, H, W), dtype=np.int32)

threads_per_block = (8, 8, 8)
blocks_per_grid = (
    (W + 7) // 8,
    (H + 7) // 8,
    (D + 7) // 8
)

volume_init_3d[blocks_per_grid, threads_per_block](
    d_vol, D, H, W
)

result = d_vol.copy_to_host()

z, y, x = np.indices((D, H, W))
expected = z * 10000 + y * 100 + x

np.testing.assert_array_equal(result, expected)

flat_index = 2 * H * W + 3 * W + 4

assert result.ravel()[flat_index] == result[2, 3, 4]

print("Task 5 PASSED!")
print("Volume shape:", result.shape)
print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)
print("Total elements:", result.size)
print("First 10 values:", result[0, 0, :10])
print("Value at [2, 3, 4]:", result[2, 3, 4])
print("Flattened index:", flat_index)


Task 5 PASSED!
Volume shape: (33, 47, 65)
Threads per block: (8, 8, 8)
Blocks per grid: (9, 6, 5)
Total elements: 100815
First 10 values: [0 1 2 3 4 5 6 7 8 9]
Value at [2, 3, 4]: 20304
Flattened index: 6309


In [16]:

import numpy as np
from numba import cuda

@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    x, y = cuda.grid(2)

    if y < img_rgb.shape[0] and x < img_rgb.shape[1]:
        r = img_rgb[y, x, 0]
        g = img_rgb[y, x, 1]
        b = img_rgb[y, x, 2]

        img_gray[y, x] = 0.299 * r + 0.587 * g + 0.114 * b

H = 513
W = 1021

np.random.seed(42)
img_rgb = np.random.rand(H, W, 3).astype(np.float32)

d_rgb = cuda.to_device(img_rgb)
d_gray = cuda.device_array((H, W), dtype=np.float32)

threads_per_block = (16, 16)
blocks_per_grid = (
    (W + 15) // 16,
    (H + 15) // 16
)

rgb_to_grayscale[blocks_per_grid, threads_per_block](
    d_rgb, d_gray
)

result = d_gray.copy_to_host()

expected = (
    0.299 * img_rgb[:, :, 0] +
    0.587 * img_rgb[:, :, 1] +
    0.114 * img_rgb[:, :, 2]
)

np.testing.assert_allclose(result, expected, atol=1e-4)

print("Task 6 PASSED!")
print("Original image shape:", img_rgb.shape)
print("Grayscale image shape:", result.shape)
print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)
print("First 5 grayscale values:", result[0, :5])


Task 6 PASSED!
Original image shape: (513, 1021, 3)
Grayscale image shape: (513, 1021)
Threads per block: (16, 16)
Blocks per grid: (64, 33)
First 5 grayscale values: [0.7535041  0.28836522 0.5943395  0.33436653 0.39427146]


In [17]:

import numpy as np
from numba import cuda

@cuda.jit
def vector_lerp(a, b, t, out):
    i = cuda.grid(1)

    if i < a.size:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]

N = 1000000

np.random.seed(42)

a = np.random.rand(N).astype(np.float32)
b = np.random.rand(N).astype(np.float32)
t = np.random.rand(N).astype(np.float32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_t = cuda.to_device(t)
d_out = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (N + 255) // 256

vector_lerp[blocks_per_grid, threads_per_block](
    d_a, d_b, d_t, d_out
)

result = d_out.copy_to_host()
expected = (1.0 - t) * a + t * b

np.testing.assert_allclose(result, expected, atol=1e-5)

print("Task 7 PASSED!")
print("Total elements:", N)
print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)
print("First 10 results:", result[:10])


Task 7 PASSED!
Total elements: 1000000
Threads per block: 256
Blocks per grid: 3907
First 10 results: [0.39700136 0.77592105 0.26944932 0.58229    0.31936786 0.5282593
 0.5306975  0.28857866 0.1814822  0.7336829 ]


In [18]:

import numpy as np
from numba import cuda

@cuda.jit
def vector_threshold(x, thresh, mask):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        mask[i] = x[i] >= thresh
        i += stride

N = 1000000
thresh = 0.5

np.random.seed(42)
x = np.random.rand(N).astype(np.float32)

d_x = cuda.to_device(x)
d_mask = cuda.device_array(N, dtype=np.uint8)

threads_per_block = 256
blocks_per_grid = 64

vector_threshold[blocks_per_grid, threads_per_block](
    d_x, thresh, d_mask
)

result = d_mask.copy_to_host()
expected = (x >= thresh).astype(np.uint8)

np.testing.assert_array_equal(result, expected)

print("Task 8 PASSED!")
print("Total elements:", N)
print("Threshold:", thresh)
print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)
print("First 10 input values:", x[:10])
print("First 10 mask values:", result[:10])
print("Total ones:", np.sum(result))
print("Total zeros:", np.sum(result == 0))


Task 8 PASSED!
Total elements: 1000000
Threshold: 0.5
Threads per block: 256
Blocks per grid: 64
First 10 input values: [0.37454012 0.9507143  0.7319939  0.5986585  0.15601864 0.15599452
 0.05808361 0.8661761  0.601115   0.7080726 ]
First 10 mask values: [0 1 1 1 0 0 0 1 1 1]
Total ones: 500517
Total zeros: 499483


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
